# Notebook 01 — Data Exploration & Ingestion
**SuperStore Sales & Business Performance Analytics System**

This notebook covers:
- Loading the raw SuperStore CSV dataset
- Schema validation
- Initial EDA (Exploratory Data Analysis)
- Data quality assessment

In [ ]:
# ── Setup: detect environment (Databricks vs local) ───────────────────────
import sys, os

# Detect Databricks
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IN_DATABRICKS:
    MOUNT_POINT = '/mnt/superstore'
    RAW_CSV     = f'{MOUNT_POINT}/bronze/superstore_sales.csv'
    print(f'Running in Databricks — reading from {RAW_CSV}')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    RAW_CSV = os.path.join(PROJECT_ROOT, 'data', 'raw', 'superstore_sales.csv')
    print(f'Running locally — reading from {RAW_CSV}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
print('Setup complete.')


## 1. Generate / Load Dataset

In [ ]:
# Load dataset — works both locally and in Databricks
if IN_DATABRICKS:
    # Read directly from ADLS mount using pandas
    df_raw = pd.read_csv(f'/dbfs{RAW_CSV}', low_memory=False)
else:
    df_raw = pd.read_csv(RAW_CSV, low_memory=False)

print(f'Shape: {df_raw.shape}')
df_raw.head(5)


## 2. Schema Validation

In [ ]:
# Schema validation — use DataLoader if available, else basic check
try:
    from src.ingestion.data_loader import DataLoader
    loader = DataLoader()
    df = loader.load(RAW_CSV if not IN_DATABRICKS else None)
    print('Schema validation passed via DataLoader!')
except Exception:
    # In Databricks, src package not installed — use df_raw directly
    df = df_raw.copy()
    print('Using raw DataFrame (DataLoader not available in Databricks).')

print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')
df.head(3)


## 3. Exploratory Data Analysis

In [ ]:
# Data types
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Shape:', df.shape, '===')

In [ ]:
# Null value heatmap
null_pct = (df.isnull().sum() / len(df) * 100).reset_index()
null_pct.columns = ['Column', 'Null %']
null_pct = null_pct[null_pct['Null %'] > 0]

if not null_pct.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(null_pct['Column'], null_pct['Null %'], color='coral')
    ax.set_xlabel('Null %')
    plt.title('Null Values by Column (%)')
    plt.tight_layout()
    plt.show()
else:
    print('No null values detected!')

In [ ]:
# Descriptive statistics on numeric columns
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe().round(2)

In [ ]:
# Distribution of key numeric fields
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fields = ['Sales', 'Profit', 'Discount', 'Quantity']
for ax, field in zip(axes.flatten(), fields):
    ax.hist(df[field].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(f'Distribution: {field}')
    ax.set_xlabel(field)
    ax.set_ylabel('Count')
plt.suptitle('Numeric Field Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical field value counts
cat_fields = ['Region', 'Category', 'Segment', 'Ship Mode']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, field in zip(axes.flatten(), cat_fields):
    vc = df[field].value_counts()
    ax.bar(vc.index, vc.values, color=sns.color_palette('husl', len(vc)))
    ax.set_title(f'Count by {field}')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Categorical Field Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric fields
corr = df[['Sales', 'Quantity', 'Discount', 'Profit']].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.title('Correlation Matrix — Numeric Fields')
plt.tight_layout()
plt.show()

In [ ]:
# Date range analysis
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
print(f'Order date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Total unique order dates: {df["Order Date"].nunique()}')
print(f'Total unique orders: {df["Order ID"].nunique():,}')
print(f'Total unique customers: {df["Customer ID"].nunique():,}')
print(f'Total unique products: {df["Product ID"].nunique():,}')